# ECC Tensile Property Prediction: Complete Pipeline
## Forward Pass (Stress & Strain) + Inverse Design (Cost-Optimal Mix)

**Dataset:** 560 tension test specimens, 285 unique mix designs  
**Features:** 18 original + 19 engineered = 37 total  
**Targets:** Second Stress (MPa), Second Strain (mm/mm)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor, 
                               RandomForestRegressor, ExtraTreesClassifier)
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy import stats

print("All imports loaded.")

## 1. Data Loading and Preprocessing

In [ ]:
# Load cleaned dataset
df = pd.read_csv('ECC_An_Final_with_Cost.csv')
features = df.columns[:-2].tolist()
X_df = df[features].copy()
y_strain = df['Second Strain'].values
y_stress = df['Second Stress'].values

print(f"Dataset: {len(df)} specimens, {len(features)} original features")
print(f"Targets: Second Stress ({y_stress.min():.2f} - {y_stress.max():.2f} MPa)")
print(f"         Second Strain ({y_strain.min()*100:.3f} - {y_strain.max()*100:.2f} %)")
print(f"\nOriginal features: {features}")

## 2. Feature Engineering (37 total)

### Sources:
- **8 data-driven features:** Paste volume, SCM ratio, Aggregate ratio, Fiber surface area, etc.
- **2 UTRGV features (Park et al., 2020):** FA/Binder ratio, Sand/Binder ratio
- **9 Li (2003) micromechanics proxies:** sigma_cu, sigma_crack, PSH indices, etc.

In [ ]:
# Binder and paste calculations
binder = X_df['Cement'] + X_df['Fly ash F'] + X_df['Fly ash C'] + X_df['GGBS'] + X_df['Silica Fume']
paste = X_df['Cement'] + X_df['Water'] + X_df['Fly ash F'] + X_df['GGBS'] + X_df['Silica Fume'] + X_df['Fly ash C']

# --- Data-driven features (8) ---
X_df['Paste_volume'] = paste
X_df['SCM_ratio'] = (X_df['Fly ash F'] + X_df['GGBS'] + X_df['Silica Fume'] + X_df['Fly ash C']) / (X_df['Cement'] + 1e-8)
X_df['Aggregate_ratio'] = (X_df['Sand'] + X_df['Coarse Aggr.']) / (paste + 1e-8)
X_df['Fiber_surface_area'] = X_df['Fiber Volume'] * X_df['Length (mm)'] / (X_df['Diameter (mm)'] + 1e-8)
X_df['FiberVol_x_LD'] = X_df['Fiber Volume'] * X_df['L/D']
X_df['has_GGBS'] = (X_df['GGBS'] > 0).astype(int)
X_df['has_CoarseAggr'] = (X_df['Coarse Aggr.'] > 0).astype(int)
X_df['has_SilicaFume'] = (X_df['Silica Fume'] > 0).astype(int)

# --- UTRGV (Park et al., 2020) features (2) ---
X_df['FA_binder_ratio'] = (X_df['Fly ash F'] + X_df['Fly ash C']) / (binder + 1e-8)
X_df['S_B_ratio'] = X_df['Sand'] / (binder + 1e-8)

# --- Li (2003) micromechanics proxies (9) ---
WB = X_df['W/B'].values + 1e-8
X_df['sigma_cu_proxy'] = X_df['Fiber Volume'].values * X_df['L/D'].values * (1.0 / WB)
X_df['sigma_crack_proxy'] = (binder.values / WB) / np.sqrt(
    (X_df['Sand'].values + X_df['Coarse Aggr.'].values) / (paste.values + 1e-8) + WB + 1e-8)
X_df['PSH_strength'] = X_df['sigma_cu_proxy'] / (X_df['sigma_crack_proxy'] + 1e-8)
X_df['Jb_complement'] = X_df['sigma_cu_proxy'] * (X_df['Length (mm)'].values / 2)
X_df['J_tip_proxy'] = X_df['sigma_crack_proxy'] ** 2
X_df['PSH_energy'] = X_df['Jb_complement'] / (X_df['J_tip_proxy'] + 1e-8)
d_f = X_df['Diameter (mm)'].values + 1e-8
L_f = X_df['Length (mm)'].values
V_f = X_df['Fiber Volume'].values
X_df['fiber_efficiency'] = (V_f / (np.pi * (d_f/2)**2 * L_f + 1e-8)) * L_f * np.pi * d_f
X_df['tau_proxy'] = 1.0 / WB
X_df['flaw_size_proxy'] = (X_df['Sand'].values + X_df['Coarse Aggr.'].values) / (paste.values + 1e-8) + WB

all_feature_names = list(X_df.columns)
X_arr = X_df.values

print(f"Total features: {X_arr.shape[1]}")
print(f"  Original: 18")
print(f"  Data-driven: 8")
print(f"  UTRGV (Park 2020): 2")
print(f"  Li (2003) proxies: 9")
print(f"  Total: {X_arr.shape[1]}")

## 3. Forward Pass: Model Training and Evaluation

### Models:
- **Stress:** Extra Trees Regressor (max_depth=20, all features)
- **Strain:** Weighted blend of Extra Trees (0.7) + Gradient Boosting (0.3)
- **CQR intervals:** Random Forest (500 trees) with conformal calibration

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- STRESS ---
print("=" * 60)
print("STRESS MODEL: Extra Trees")
print("=" * 60)

oof_stress = cross_val_predict(
    ExtraTreesRegressor(n_estimators=500, max_depth=20, min_samples_leaf=2, 
                         max_features=1.0, random_state=42, n_jobs=-1),
    X_arr, y_stress, cv=kf)

r2_s = r2_score(y_stress, oof_stress)
sp_s = stats.spearmanr(y_stress, oof_stress)[0]
mae_s = mean_absolute_error(y_stress, oof_stress)
rmse_s = np.sqrt(mean_squared_error(y_stress, oof_stress))

print(f"  R-squared:  {r2_s:.4f}")
print(f"  Spearman:   {sp_s:.4f}")
print(f"  MAE:        {mae_s:.3f} MPa")
print(f"  RMSE:       {rmse_s:.3f} MPa")

# --- STRAIN ---
print(f"\n{'=' * 60}")
print("STRAIN MODEL: ET (0.7) + GBR (0.3) Blend")
print("=" * 60)

oof_et = cross_val_predict(
    ExtraTreesRegressor(n_estimators=500, max_depth=15, min_samples_leaf=2, 
                         max_features='sqrt', random_state=42, n_jobs=-1),
    X_arr, y_strain, cv=kf)

oof_gbr = cross_val_predict(
    GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, 
                               subsample=0.85, loss='squared_error', random_state=42),
    X_arr, y_strain, cv=kf)

oof_strain = 0.7 * oof_et + 0.3 * oof_gbr

r2_e = r2_score(y_strain, oof_strain)
sp_e = stats.spearmanr(y_strain, oof_strain)[0]
mae_e = mean_absolute_error(y_strain, oof_strain)
rmse_e = np.sqrt(mean_squared_error(y_strain, oof_strain))

print(f"  R-squared:  {r2_e:.4f}")
print(f"  Spearman:   {sp_e:.4f}")
print(f"  MAE:        {mae_e*100:.3f} %")
print(f"  RMSE:       {rmse_e*100:.3f} %")

### 3.1 Predicted vs Actual Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Stress scatter
ax = axes[0]
ax.scatter(y_stress, oof_stress, s=12, c='#4472C4', alpha=0.5, edgecolors='none')
mn, mx = min(y_stress.min(), oof_stress.min())-0.3, max(y_stress.max(), oof_stress.max())+0.3
ax.plot([mn,mx],[mn,mx],'k--',lw=1,alpha=0.5)
ax.set_xlabel('Actual Stress (MPa)', fontsize=11)
ax.set_ylabel('Predicted Stress (MPa)', fontsize=11)
ax.set_title(f'Stress: R\u00b2 = {r2_s:.4f}, \u03c1 = {sp_s:.4f}', fontsize=12)
ax.set_xlim(mn,mx); ax.set_ylim(mn,mx); ax.set_aspect('equal')
ax.grid(True, alpha=0.15)

# Strain scatter
ax = axes[1]
ax.scatter(y_strain*100, oof_strain*100, s=12, c='#C55A11', alpha=0.5, edgecolors='none')
mn2, mx2 = -0.5, max(y_strain.max(), oof_strain.max())*100+1
ax.plot([0,mx2],[0,mx2],'k--',lw=1,alpha=0.5)
ax.set_xlabel('Actual Strain (%)', fontsize=11)
ax.set_ylabel('Predicted Strain (%)', fontsize=11)
ax.set_title(f'Strain: R\u00b2 = {r2_e:.4f}, \u03c1 = {sp_e:.4f}', fontsize=12)
ax.set_xlim(0,mx2); ax.set_ylim(0,mx2); ax.set_aspect('equal')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

### 3.2 CQR Prediction Intervals

In [ ]:
n = len(X_arr)

def compute_cqr(X, y, kf):
    oof_lo = np.full(len(X), np.nan)
    oof_hi = np.full(len(X), np.nan)
    for tr, te in kf.split(X):
        cs = len(tr) // 4
        perm = np.random.RandomState(42).permutation(len(tr))
        cal = tr[perm[:cs]]; fit = tr[perm[cs:]]
        rf = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_leaf=2, 
                                    max_features=0.7, random_state=42, n_jobs=-1)
        rf.fit(X[fit], y[fit])
        tp_cal = np.array([t.predict(X[cal]) for t in rf.estimators_])
        lo_c = np.percentile(tp_cal, 10, axis=0)
        hi_c = np.percentile(tp_cal, 90, axis=0)
        scores = np.maximum(lo_c - y[cal], y[cal] - hi_c)
        q = np.quantile(scores, 0.80)
        tp_te = np.array([t.predict(X[te]) for t in rf.estimators_])
        oof_lo[te] = np.percentile(tp_te, 10, axis=0) - q
        oof_hi[te] = np.percentile(tp_te, 90, axis=0) + q
    return np.maximum(oof_lo, 0), oof_hi

strain_lo, strain_hi = compute_cqr(X_arr, y_strain, kf)
stress_lo, stress_hi = compute_cqr(X_arr, y_stress, kf)

for name, y_t, lo, hi, unit, mult in [
    ("Stress", y_stress, stress_lo, stress_hi, "MPa", 1),
    ("Strain", y_strain, strain_lo, strain_hi, "%", 100)]:
    cov = ((y_t >= lo) & (y_t <= hi)).mean()
    w = (hi - lo).mean()
    print(f"{name}: Coverage = {cov:.3f}, Avg Width = {w*mult:.2f} {unit}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name, y_t, lo, hi, pred, unit, mult, color in [
    (axes[0], "Stress", y_stress, stress_lo, stress_hi, oof_stress, "MPa", 1, '#4472C4'),
    (axes[1], "Strain", y_strain, strain_lo, strain_hi, oof_strain, "%", 100, '#C55A11')]:
    order = np.argsort(y_t)
    cov = ((y_t >= lo) & (y_t <= hi)).mean()
    ax.fill_between(range(n), lo[order]*mult, hi[order]*mult, alpha=0.25, color=color, 
                     label=f'80% CQR (coverage={cov:.1%})')
    ax.scatter(range(n), y_t[order]*mult, s=6, c='#333', alpha=0.5, label='Actual', zorder=3)
    ax.scatter(range(n), pred[order]*mult, s=6, c=color, alpha=0.3, label='Predicted', zorder=2)
    ax.set_xlabel(f'Samples (sorted by actual {name.lower()})')
    ax.set_ylabel(f'{name} ({unit})')
    ax.set_title(f'{name}: CQR Interval')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

## 4. Train Production Models (on full dataset)

These models are used by the inverse design pipeline.

In [ ]:
# Train on ALL data for production use
m_stress = ExtraTreesRegressor(n_estimators=500, max_depth=20, min_samples_leaf=2, 
                                max_features=1.0, random_state=42, n_jobs=-1)
m_stress.fit(X_arr, y_stress)

m_strain_et = ExtraTreesRegressor(n_estimators=500, max_depth=15, min_samples_leaf=2, 
                                   max_features='sqrt', random_state=42, n_jobs=-1)
m_strain_et.fit(X_arr, y_strain)

m_strain_gbr = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, 
                                          subsample=0.85, loss='squared_error', random_state=42)
m_strain_gbr.fit(X_arr, y_strain)

# RF for prediction intervals
m_rf_strain = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_leaf=2, 
                                     max_features=0.7, random_state=42, n_jobs=-1)
m_rf_strain.fit(X_arr, y_strain)

m_rf_stress = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_leaf=2, 
                                     max_features=0.7, random_state=42, n_jobs=-1)
m_rf_stress.fit(X_arr, y_stress)

print("Production models trained:")
print(f"  Stress: Extra Trees (R2={r2_s:.4f})")
print(f"  Strain: ET+GBR blend (R2={r2_e:.4f})")
print(f"  Intervals: RF for both targets")

## 5. Inverse Design: Target Performance to Cheapest Mix

### Method: Random Sampling + Local Refinement
1. Generate 100,000 random mixes within realistic bounds
2. Predict stress and strain for all (batch prediction, ~3 seconds)
3. Filter to mixes meeting target ranges, sort by cost
4. Refine top 50 candidates with 5% perturbations
5. Return top 3 cheapest with prediction intervals

In [ ]:
# Material prices (USD per kg)
prices = {
    'Cement': 0.30420, 'Water': 0.00220, 'Sand': 0.01980,
    'Fly ash C': 0.01349, 'Fly ash F': 0.01349,
    'GGBS': 0.07860, 'Coarse Aggr.': 0.01980,
    'Silica Fume': 0.39160, 'Water Reducer / SP': 3.91600
}

# Optimization variables (9 ingredients the engineer controls)
mix_cols = ['Cement', 'Water', 'Sand', 'Fly ash C', 'Fly ash F', 
            'GGBS', 'Coarse Aggr.', 'Silica Fume', 'Water Reducer / SP']

# Realistic bounds from dataset
mix_bounds = np.array([
    [150, 1000],   # Cement
    [80, 500],     # Water
    [0, 1000],     # Sand
    [0, 800],      # Fly ash C
    [0, 1000],     # Fly ash F
    [0, 800],      # GGBS
    [0, 1000],     # Coarse Aggr.
    [0, 200],      # Silica Fume
    [0, 40],       # Water Reducer / SP
], dtype=float)


def build_feature_batch(mixes, fiber_params):
    """Build full 37-feature matrix from Nx9 mix array + fiber params (vectorized)."""
    N = len(mixes)
    fv, lf, df_, ld, ri, fk = fiber_params
    cem, wat, sand, fa_c, fa_f = mixes[:,0], mixes[:,1], mixes[:,2], mixes[:,3], mixes[:,4]
    ggbs, ca, sf, sp = mixes[:,5], mixes[:,6], mixes[:,7], mixes[:,8]
    
    bnd = cem + fa_f + fa_c + ggbs + sf
    bnd_s = np.maximum(bnd, 50)
    wb = wat / bnd_s
    cb = cem / bnd_s
    wc = wat / (cem + 1e-8)
    pst = cem + wat + fa_f + ggbs + sf + fa_c
    ar = (sand + ca) / (pst + 1e-8)
    scu = fv * ld * (1.0 / (wb + 1e-8))
    scr = (bnd_s / (wb + 1e-8)) / np.sqrt(ar + wb + 1e-8)
    
    X = np.column_stack([
        np.full(N, fv), np.full(N, lf), np.full(N, df_), np.full(N, ld), np.full(N, ri),
        cem, wat, sand, fa_c, fa_f, ggbs, ca, sf, sp, np.full(N, fk),
        cb, wc, wb, pst,
        (fa_f + ggbs + sf + fa_c) / (cem + 1e-8), ar,
        fv * lf / (df_ + 1e-8) * np.ones(N), fv * ld * np.ones(N),
        (ggbs > 0).astype(float), (ca > 0).astype(float), (sf > 0).astype(float),
        (fa_f + fa_c) / (bnd_s + 1e-8), sand / (bnd_s + 1e-8),
        scu, scr, scu / (scr + 1e-8), ar + wb
    ])
    
    valid = (bnd >= 50) & (wb >= 0.05) & (wb <= 1.0)
    return X, valid


def compute_costs(mixes):
    """Compute cost per m3 for Nx9 array."""
    price_vec = np.array([prices.get(c, 0) for c in mix_cols])
    return mixes @ price_vec


def inverse_design(stress_range, strain_range_pct, fiber_params, n_samples=100000):
    """
    Find top 3 cost-optimal mixes for target performance.
    
    Parameters:
        stress_range: (min, max) in MPa
        strain_range_pct: (min, max) in percent
        fiber_params: (Vf, Length_mm, Diameter_mm, L_D, RI, Fiber_kg)
    
    Returns: list of 3 dicts with mix, cost, predictions, intervals
    """
    s_min, s_max = stress_range
    e_min, e_max = strain_range_pct[0] / 100, strain_range_pct[1] / 100
    
    # Phase 1: Random sampling
    print(f"Phase 1: Sampling {n_samples:,} random mixes...")
    rng = np.random.RandomState(42)
    mixes = rng.uniform(mix_bounds[:, 0], mix_bounds[:, 1], size=(n_samples, 9))
    
    X, valid = build_feature_batch(mixes, fiber_params)
    stress_pred = m_stress.predict(X)
    strain_pred = 0.7 * m_strain_et.predict(X) + 0.3 * m_strain_gbr.predict(X)
    costs = compute_costs(mixes)
    
    meets = valid & (stress_pred >= s_min) & (stress_pred <= s_max) & \
            (strain_pred >= e_min) & (strain_pred <= e_max)
    print(f"  Valid: {valid.sum():,}, Meet targets: {meets.sum():,}")
    
    if meets.sum() < 3:
        print("  Relaxing targets...")
        penalty = np.maximum(0, s_min - stress_pred) * 100 + \
                  np.maximum(0, stress_pred - s_max) * 50 + \
                  np.maximum(0, e_min - strain_pred) * 10000 + \
                  np.maximum(0, strain_pred - e_max) * 5000
        penalty[~valid] = 1e9
        top_idx = np.argsort(costs + penalty)[:200]
    else:
        valid_idx = np.where(meets)[0]
        top_idx = valid_idx[np.argsort(costs[valid_idx])[:200]]
    
    # Phase 2: Local refinement
    print("Phase 2: Refining top candidates...")
    refined = []
    for idx in top_idx[:50]:
        noise = rng.normal(1.0, 0.05, size=(100, 9)) 
        perturbed = np.clip(mixes[idx] * noise, mix_bounds[:, 0], mix_bounds[:, 1])
        refined.append(perturbed)
    
    refined = np.vstack(refined)
    X_ref, valid_ref = build_feature_batch(refined, fiber_params)
    stress_ref = m_stress.predict(X_ref)
    strain_ref = 0.7 * m_strain_et.predict(X_ref) + 0.3 * m_strain_gbr.predict(X_ref)
    costs_ref = compute_costs(refined)
    meets_ref = valid_ref & (stress_ref >= s_min) & (stress_ref <= s_max) & \
                (strain_ref >= e_min) & (strain_ref <= e_max)
    
    all_m = np.vstack([mixes[top_idx], refined])
    all_s = np.concatenate([stress_pred[top_idx], stress_ref])
    all_e = np.concatenate([strain_pred[top_idx], strain_ref])
    all_c = np.concatenate([costs[top_idx], costs_ref])
    all_v = np.concatenate([meets[top_idx] if meets.sum() >= 3 else np.ones(len(top_idx), bool), meets_ref])
    
    print(f"  Total candidates: {len(all_m):,}, Valid: {all_v.sum():,}")
    
    # Select top 3
    if all_v.sum() >= 3:
        vi = np.where(all_v)[0]
        top3_i = vi[np.argsort(all_c[vi])[:3]]
    else:
        pen = np.maximum(0, s_min - all_s)*100 + np.maximum(0, all_s - s_max)*50 + \
              np.maximum(0, e_min - all_e)*10000 + np.maximum(0, all_e - e_max)*5000
        top3_i = np.argsort(all_c + pen)[:3]
    
    # Phase 3: Intervals for top 3
    print("Phase 3: Computing prediction intervals...")
    results = []
    for i in top3_i:
        mix = all_m[i]
        X_single, _ = build_feature_batch(mix.reshape(1, -1), fiber_params)
        st_trees = np.array([t.predict(X_single)[0] for t in m_rf_stress.estimators_])
        sn_trees = np.array([t.predict(X_single)[0] for t in m_rf_strain.estimators_])
        
        bnd = mix[0] + mix[3] + mix[4] + mix[5] + mix[7]
        results.append({
            'mix': {c: v for c, v in zip(mix_cols, mix)},
            'cost': all_c[i],
            'stress': all_s[i], 'stress_lo': np.percentile(st_trees, 10), 'stress_hi': np.percentile(st_trees, 90),
            'strain': all_e[i], 'strain_lo': max(0, np.percentile(sn_trees, 10)), 'strain_hi': np.percentile(sn_trees, 90),
            'W_B': mix[1] / bnd if bnd > 0 else 0,
            'FA_Binder': (mix[3] + mix[4]) / bnd if bnd > 0 else 0,
        })
    
    return results


def print_results(results):
    """Display inverse design results."""
    for i, s in enumerate(results):
        print(f"\n  {'='*50}")
        print(f"  MIX #{i+1}  |  Cost: ${s['cost']:.2f}/m3")
        print(f"  {'='*50}")
        print(f"  Stress: {s['stress']:.2f} MPa  (interval: {s['stress_lo']:.2f} - {s['stress_hi']:.2f})")
        print(f"  Strain: {s['strain']*100:.2f}%    (interval: {s['strain_lo']*100:.2f} - {s['strain_hi']*100:.2f}%)")
        print(f"  W/B = {s['W_B']:.3f}  |  FA/Binder = {s['FA_Binder']:.3f}")
        print(f"  Ingredients:")
        for c in mix_cols:
            v = s['mix'][c]
            if v > 0.5:
                uc = v * prices.get(c, 0)
                print(f"    {c:<25} {v:>8.1f} kg/m3    (${uc:.2f})")


print("Inverse design functions defined.")

## 6. Run Inverse Design

Specify your target stress range, strain range, and fiber type. The optimizer finds the 3 cheapest mixes meeting both targets.

In [ ]:
# === USER INPUTS ===
TARGET_STRESS = (3.0, 6.0)    # MPa range
TARGET_STRAIN = (2.0, 5.0)    # % range

# PVA fiber (most common in dataset)
# Format: (Vf, Length_mm, Diameter_mm, L_D, RI, Fiber_kg_per_m3)
PVA_FIBER = (0.02, 12.0, 0.039, 307.69, 6.15, 26.0)

print("=" * 60)
print("INVERSE DESIGN")
print("=" * 60)
print(f"Target stress: {TARGET_STRESS[0]} - {TARGET_STRESS[1]} MPa")
print(f"Target strain: {TARGET_STRAIN[0]} - {TARGET_STRAIN[1]} %")
print(f"Fiber: PVA (L=12mm, d=0.039mm, Vf=2%)\n")

top3 = inverse_design(TARGET_STRESS, TARGET_STRAIN, PVA_FIBER)
print_results(top3)

## 7. Monte Carlo Validation

Simulate batching errors AND model prediction uncertainty to get honest reliability estimates.

In [ ]:
def monte_carlo_validate(recipe_mix, fiber_params, target_stress, target_strain_pct, 
                         n_sim=3000, error_margins=[0.03, 0.05, 0.10]):
    """
    Monte Carlo validation with HONEST uncertainty (model error + batching noise).
    
    Parameters:
        recipe_mix: array of 9 ingredient quantities
        target_stress: (min, max) MPa
        target_strain_pct: (min, max) percent
        error_margins: list of batching error levels (0.03 = 3%)
    """
    base = np.array(recipe_mix)
    STRESS_RMSE = rmse_s   # from cross-validation
    STRAIN_RMSE = rmse_e   # from cross-validation
    rng = np.random.RandomState(42)
    
    fig, axes = plt.subplots(2, len(error_margins), figsize=(6*len(error_margins), 10))
    if len(error_margins) == 1:
        axes = axes.reshape(-1, 1)
    
    print(f"Monte Carlo Validation ({n_sim} simulations per error level)")
    print(f"Model RMSE: Stress = {STRESS_RMSE:.3f} MPa, Strain = {STRAIN_RMSE*100:.3f}%")
    print(f"{'Error':>8} {'Stress OK':>10} {'Strain OK':>10} {'Both OK':>10} {'Verdict':>12}")
    print("-" * 55)
    
    for col, err in enumerate(error_margins):
        noise = rng.normal(1.0, err/3, size=(n_sim, 9))
        perturbed = np.maximum(base * noise, 0)
        X_batch = build_feature_batch(perturbed, fiber_params)[0]
        
        pred_stress = m_stress.predict(X_batch)
        pred_strain = 0.7 * m_strain_et.predict(X_batch) + 0.3 * m_strain_gbr.predict(X_batch)
        
        # Add model uncertainty
        real_stress = pred_stress + rng.normal(0, STRESS_RMSE, n_sim)
        real_strain = np.maximum(pred_strain + rng.normal(0, STRAIN_RMSE, n_sim), 0)
        
        s_ok = ((real_stress >= target_stress[0]) & (real_stress <= target_stress[1])).mean() * 100
        e_ok = ((real_strain >= target_strain_pct[0]/100) & (real_strain <= target_strain_pct[1]/100)).mean() * 100
        b_ok = ((real_stress >= target_stress[0]) & (real_stress <= target_stress[1]) & 
                (real_strain >= target_strain_pct[0]/100) & (real_strain <= target_strain_pct[1]/100)).mean() * 100
        
        verdict = "ROBUST" if b_ok >= 85 else "MODERATE" if b_ok >= 60 else "MARGINAL" if b_ok >= 40 else "UNSTABLE"
        print(f"{err*100:>7.0f}% {s_ok:>9.1f}% {e_ok:>9.1f}% {b_ok:>9.1f}% {verdict:>12}")
        
        # Stress plot
        ax = axes[0, col]
        ax.hist(pred_stress, bins=40, alpha=0.5, color='#4472C4', label='Model only', density=True)
        ax.hist(real_stress, bins=40, alpha=0.5, color='#C55A11', label='+ model uncertainty', density=True)
        ax.axvline(target_stress[0], color='red', ls='--', lw=1.5)
        ax.axvline(target_stress[1], color='red', ls='--', lw=1.5)
        ax.set_title(f'Stress (+/-{err*100:.0f}% batch error)', fontweight='bold')
        ax.set_xlabel('Stress (MPa)')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.15)
        
        # Strain plot
        ax = axes[1, col]
        ax.hist(pred_strain*100, bins=40, alpha=0.5, color='#4472C4', label='Model only', density=True)
        ax.hist(real_strain*100, bins=40, alpha=0.5, color='#C55A11', label='+ model uncertainty', density=True)
        ax.axvline(target_strain_pct[0], color='red', ls='--', lw=1.5)
        ax.axvline(target_strain_pct[1], color='red', ls='--', lw=1.5)
        ax.set_title(f'Strain (+/-{err*100:.0f}% batch error)', fontweight='bold')
        ax.set_xlabel('Strain (%)')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.15)
    
    fig.suptitle('Monte Carlo Validation\nBlue = model spread | Orange = realistic uncertainty', 
                  fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Run on Mix #1
mix1 = list(top3[0]['mix'].values())
monte_carlo_validate(mix1, PVA_FIBER, TARGET_STRESS, TARGET_STRAIN)

## 8. How to Use

### Change targets:
```python
# Example: High ductility ECC
top3 = inverse_design(
    stress_range=(2.0, 5.0),      # MPa
    strain_range_pct=(3.0, 8.0),  # percent
    fiber_params=PVA_FIBER
)
print_results(top3)
```

### Change fiber type:
```python
# PE fiber (high L/D)
PE_FIBER = (0.02, 18.0, 0.026, 692.31, 13.85, 26.0)
top3 = inverse_design((3.0, 6.0), (2.0, 5.0), PE_FIBER)
```

### Validate a custom recipe:
```python
my_recipe = [400, 300, 450, 0, 600, 0, 0, 0, 5]  # 9 ingredients
monte_carlo_validate(my_recipe, PVA_FIBER, (3.0, 6.0), (2.0, 5.0))
```

### Forward prediction for a single mix:
```python
mix = np.array([400, 300, 450, 0, 600, 0, 0, 0, 5]).reshape(1, -1)
X_feat, valid = build_feature_batch(mix, PVA_FIBER)
stress = m_stress.predict(X_feat)[0]
strain = 0.7 * m_strain_et.predict(X_feat)[0] + 0.3 * m_strain_gbr.predict(X_feat)[0]
print(f"Stress: {stress:.2f} MPa, Strain: {strain*100:.2f}%")
```